In [ ]:
import requests
from bs4 import BeautifulSoup

url="https://www.globalfirepower.com/countries-listing.php"

page = requests.get(url)

soup = BeautifulSoup(page.text, "html")
soup

In [14]:
#name
import pandas as pd

name_span = soup.find_all('span', class_='textWhite textLarge textShadow')

data = []

for i in range(0, len(name_span)-1, 2):
    full_name = name_span[i].text.strip()
    short_name = name_span[i+1].text.strip()

    data.append({
        "Country_Code": short_name,      # USA
        "Country_Name": full_name        # United States of America
    })

df = pd.DataFrame(data)

print(df)

    Country_Code              Country_Name
0            USA             United States
1            RUS                    Russia
2            CHN                     China
3            IND                     India
4            SKO               South Korea
..           ...                       ...
140          LIB                   Liberia
141          SRN                  Suriname
142          CAR  Central African Republic
143          BLZ                     Beliz
144          BUT                    Bhutan

[145 rows x 2 columns]


In [50]:
import requests
from bs4 import BeautifulSoup

url="https://www.globalfirepower.com/total-population-by-country.php"

page = requests.get(url)

soup = BeautifulSoup(page.text, "html")

data = []

country_code_scraped = soup.find_all('div', class_='shortFormName')
total_population_c = soup.find_all('div', class_='valueContainer')

for i in range(len(country_code_scraped)):
    code = country_code_scraped[i].text.strip()
    population = total_population_c[i].text.strip()

    data.append({
        "Country_Code": code,
        "total_population": population
    })

temp_df = pd.DataFrame(data)

print(temp_df)

# Merge based on Country_Code
df = df.merge(temp_df, on="Country_Code", how="left")

print(df)




    Country_Code total_population
0            CHN    1,415,043,270
1            IND    1,409,128,296
2            USA      341,963,408
3            INO      281,562,465
4            PAK      252,363,571
..           ...              ...
140          LUX          671,254
141          SRN          646,758
142          MTN          599,849
143          BLZ          415,789
144          ICE          364,036

[145 rows x 2 columns]
    Country_Code              Country_Name total_population_x  \
0            USA             United States        150,463,900   
1            RUS                    Russia         69,002,197   
2            CHN                     China        764,123,366   
3            IND                     India        662,290,299   
4            SKO               South Korea         26,040,900   
..           ...                       ...                ...   
140          LIB                   Liberia          2,392,390   
141          SRN                  Suriname      

In [56]:
def scrape_metric(url, column_name):

    page = requests.get(url)
    soup = BeautifulSoup(page.text, "html.parser")

    data = []

    country_codes = soup.find_all('div', class_='shortFormName')
    values = soup.find_all('div', class_='valueContainer')

    for i in range(len(country_codes)):
        code = country_codes[i].text.strip()
        value = values[i].text.strip()

        data.append({
            "Country_Code": code,
            column_name: value
        })

    return pd.DataFrame(data)

In [58]:
scrape_metric('https://www.globalfirepower.com/armor-tanks-total.php', 'column_name')

,Country_Code,column_name
0,CHN,"5,870"
1,RUS,"5,630"
2,NKO,"4,895"
3,USA,"4,666"
4,IND,"3,913"
...,...,...
140,SEN,0
141,SLE,0
142,SOM,0
143,SRN,0


In [65]:
import re
import ast

with open('/content/links for military data.txt', 'r') as f:
    content = f.read()

# Extract base_url using regex
base_url_match = re.search(r"base_url:\s*'([^']+)'", content)
base_url = base_url_match.group(1) if base_url_match else None

print("base_url:", base_url)

# Extract other_sources dictionary
dict_match = re.search(r'other_sources:\s*(\{.*\})', content, re.DOTALL)
if dict_match:
    other_sources = ast.literal_eval(dict_match.group(1))
    print("\nother_sources:")
    print(other_sources)

base_url: https://www.globalfirepower.com/countries-listing.php

other_sources:
{'https://www.globalfirepower.com/total-population-by-country.php': 'total_population', 'https://www.globalfirepower.com/available-military-manpower.php': 'total_military_manpower', 'https://www.globalfirepower.com/manpower-fit-for-military-service.php': 'fit_for_service', 'https://www.globalfirepower.com/manpower-reaching-military-age-annually.php': 'population_reaching_military_age_annually', 'https://www.globalfirepower.com/active-military-manpower.php': 'active_personnel', 'https://www.globalfirepower.com/active-reserve-military-manpower.php': 'reserve_personnel', 'https://www.globalfirepower.com/manpower-paramilitary.php': 'paramilitary', 'https://www.globalfirepower.com/aircraft-total.php': 'total_military_aircraft', 'https://www.globalfirepower.com/aircraft-total-fighters.php': 'fighter_aircraft', 'https://www.globalfirepower.com/aircraft-total-attack-types.php': 'attack_aircraft', 'https://www.globa

In [67]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Function to scrape a single metric
def scrape_metric(url, column_name):
    try:
        page = requests.get(url)
        soup = BeautifulSoup(page.text, "html.parser")

        data = []

        country_codes = soup.find_all('div', class_='shortFormName')
        values = soup.find_all('div', class_='valueContainer')

        for i in range(len(country_codes)):
            code = country_codes[i].text.strip()
            value = values[i].text.strip()

            data.append({
                "Country_Code": code,
                column_name: value
            })

        return pd.DataFrame(data)

    except Exception as e:
        print(f"Error scraping {column_name}: {e}")
        return pd.DataFrame()

# Base URL
base_url = 'https://www.globalfirepower.com/countries-listing.php'

# Scrape base URL for country names
print("Scraping country names...")
page = requests.get(base_url)
soup = BeautifulSoup(page.text, "html.parser")

name_span = soup.find_all('span', class_='textWhite textLarge textShadow')

data = []
for i in range(0, len(name_span)-1, 2):
    full_name = name_span[i].text.strip()
    short_name = name_span[i+1].text.strip()

    data.append({
        "Country_Code": short_name,
        "Country_Name": full_name
    })

# Create main DataFrame
df_final = pd.DataFrame(data)
print(f"Found {len(df_final)} countries\n")

# Dictionary of other sources
other_sources = {
    'https://www.globalfirepower.com/total-population-by-country.php': 'total_population',
    'https://www.globalfirepower.com/available-military-manpower.php': 'total_military_manpower',
    'https://www.globalfirepower.com/manpower-fit-for-military-service.php': 'fit_for_service',
    'https://www.globalfirepower.com/manpower-reaching-military-age-annually.php': 'population_reaching_military_age_annually',
    'https://www.globalfirepower.com/active-military-manpower.php': 'active_personnel',
    'https://www.globalfirepower.com/active-reserve-military-manpower.php': 'reserve_personnel',
    'https://www.globalfirepower.com/manpower-paramilitary.php': 'paramilitary',
    'https://www.globalfirepower.com/aircraft-total.php': 'total_military_aircraft',
    'https://www.globalfirepower.com/aircraft-total-fighters.php': 'fighter_aircraft',
    'https://www.globalfirepower.com/aircraft-total-attack-types.php': 'attack_aircraft',
    'https://www.globalfirepower.com/aircraft-total-transports.php': 'transport_aircraft',
    'https://www.globalfirepower.com/aircraft-total-trainers.php': 'trainer_aircraft',
    'https://www.globalfirepower.com/aircraft-total-special-mission.php': 'special_mission_aircraft',
    'https://www.globalfirepower.com/aircraft-total-tanker-fleet.php': 'tanker_aircraft',
    'https://www.globalfirepower.com/aircraft-helicopters-total.php': 'total_military_helicopters',
    'https://www.globalfirepower.com/aircraft-helicopters-attack.php': 'attack_helicopters',
    'https://www.globalfirepower.com/armor-tanks-total.php': 'tanks',
    'https://www.globalfirepower.com/armor-apc-total.php': 'armored_fighting_vehicles',
    'https://www.globalfirepower.com/armor-self-propelled-guns-total.php': 'self_propelled_artillery',
    'https://www.globalfirepower.com/armor-towed-artillery-total.php': 'towed_artillery',
    'https://www.globalfirepower.com/armor-mlrs-total.php': 'rocket_projectors',
    'https://www.globalfirepower.com/navy-ships.php': 'total_naval_fleet',
    'https://www.globalfirepower.com/navy-force-by-tonnage.php': 'total_naval_fleet_tonnage_mt',
    'https://www.globalfirepower.com/navy-aircraft-carriers.php': 'aircraft_carriers',
    'https://www.globalfirepower.com/navy-helo-carriers.php': 'helicopter_carriers',
    'https://www.globalfirepower.com/navy-submarines.php': 'submarines',
    'https://www.globalfirepower.com/navy-destroyers.php': 'destroyers',
    'https://www.globalfirepower.com/navy-frigates.php': 'frigates',
    'https://www.globalfirepower.com/navy-corvettes.php': 'corvettes',
    'https://www.globalfirepower.com/navy-patrol-coastal-craft.php': 'coastal_patrol_craft',
    'https://www.globalfirepower.com/navy-mine-warfare-craft.php': 'mine_warfare_craft',
    'https://www.globalfirepower.com/defense-spending-budget.php': 'defense_budget_usd',
    'https://www.globalfirepower.com/external-debt-by-country.php': 'external_debt_usd',
    'https://www.globalfirepower.com/purchasing-power-parity.php': 'purchasing_power_parity_usd',
    'https://www.globalfirepower.com/reserves-of-foreign-exchange-and-gold.php': 'foreign_exchange_and_gold_reserves_usd',
    'https://www.globalfirepower.com/major-serviceable-airports-by-country.php': 'total_serviceable_airports',
    'https://www.globalfirepower.com/labor-force-by-country.php': 'labour_force',
    'https://www.globalfirepower.com/major-ports-and-terminals.php': 'major_ports_and_terminals',
    'https://www.globalfirepower.com/merchant-marine-strength-by-country.php': 'total_merchant_marine_fleet',
    'https://www.globalfirepower.com/railway-coverage.php': 'railway_coverage_km',
    'https://www.globalfirepower.com/roadway-coverage.php': 'roadway_coverage_km',
    'https://www.globalfirepower.com/oil-production-by-country.php': 'oil_production_bbl',
    'https://www.globalfirepower.com/oil-consumption-by-country.php': 'oil_consumption_bbl',
    'https://www.globalfirepower.com/proven-oil-reserves-by-country.php': 'proven_oil_reserves_bbl',
    'https://www.globalfirepower.com/natural-gas-production-by-country.php': 'natural_gas_production_cum',
    'https://www.globalfirepower.com/natural-gas-consumption-by-country.php': 'natural_gas_consumption_cum',
    'https://www.globalfirepower.com/proven-natural-gas-reserves-by-country.php': 'proven_natural_gas_reserves_cum',
    'https://www.globalfirepower.com/coal-production-by-country.php': 'coal_production_cum',
    'https://www.globalfirepower.com/coal-consumption-by-country.php': 'coal_consumption_mt',
    'https://www.globalfirepower.com/proven-coal-reserves-by-country.php': 'proven_coal_reserves_cum',
    'https://www.globalfirepower.com/square-land-area.php': 'total_land_area_sq_km',
    'https://www.globalfirepower.com/coastline-coverage.php': 'coastline_coverage_km',
    'https://www.globalfirepower.com/border-coverage.php': 'border_coverage_km',
    'https://www.globalfirepower.com/waterway-coverage.php': 'waterway_coverage_km'
}

print(f"Total URLs to scrape: {len(other_sources)}\n")

# Loop through other_sources and scrape each URL
counter = 1
for url, column_name in other_sources.items():
    print(f"Scraping {counter}/{len(other_sources)}: {column_name}...")

    # Scrape the metric
    df_temp = scrape_metric(url, column_name)

    # Merge if data was scraped successfully
    if not df_temp.empty:
        df_final = df_final.merge(df_temp, on='Country_Code', how='left')
        print(f"  ✓ Added {column_name}")
    else:
        print(f"  ✗ Failed to scrape {column_name}")

    # Add delay between requests
    time.sleep(1)
    counter = counter + 1

# Display results
print(f"\n{'='*50}")
print(f"Scraping complete!")
print(f"Final DataFrame shape: {df_final.shape}")
print(f"Columns: {df_final.shape[1]}")
print(f"{'='*50}\n")

print("First few rows:")
print(df_final.head())

# Save to CSV
output_file = 'global_firepower_data.csv'
df_final.to_csv(output_file, index=False)
print(f"\nData saved to '{output_file}'")

Scraping country names...
Found 145 countries

Total URLs to scrape: 54

Scraping 1/54: total_population...
  ✓ Added total_population
Scraping 2/54: total_military_manpower...
  ✓ Added total_military_manpower
Scraping 3/54: fit_for_service...
  ✓ Added fit_for_service
Scraping 4/54: population_reaching_military_age_annually...
  ✓ Added population_reaching_military_age_annually
Scraping 5/54: active_personnel...
  ✓ Added active_personnel
Scraping 6/54: reserve_personnel...
  ✓ Added reserve_personnel
Scraping 7/54: paramilitary...
  ✓ Added paramilitary
Scraping 8/54: total_military_aircraft...
  ✓ Added total_military_aircraft
Scraping 9/54: fighter_aircraft...
  ✓ Added fighter_aircraft
Scraping 10/54: attack_aircraft...
  ✓ Added attack_aircraft
Scraping 11/54: transport_aircraft...
  ✓ Added transport_aircraft
Scraping 12/54: trainer_aircraft...
  ✓ Added trainer_aircraft
Scraping 13/54: special_mission_aircraft...
  ✓ Added special_mission_aircraft
Scraping 14/54: tanker_aircra